# 14_reconciliation
- Reconciliation engine. Produces one shared table, silver.reconciliation,(extended with one extra column ):
-   recon_id | recon_type | business_date | fund_id | entity_id |
-  source_a_value | source_b_value | difference | status | break_reason |  created_at


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CASH_TOLERANCE = 0.01

def make_recon_id(recon_type, fund_id_col, entity_id_col, business_date_col):
    """Deterministic recon_id (not a random UUID) so re-running this
    notebook produces the same id for the same logical recon row -
    consistent with the idempotency approach used everywhere else in this
    project (11, 12)."""
    return F.sha2(
        F.concat_ws("||", F.lit(recon_type), fund_id_col, entity_id_col, business_date_col),
        256
    )

## A. CASH reconciliation
Both sides already sit in `silver.fund_financials` (built by
`13_financial_calculations`): `available_cash_internal` vs
`available_cash_external`. `MISSING_EXTERNAL` is determined by whether the
fund actually appears in `cash_current` at all - NOT by checking whether
`available_cash_external == 0.0`, since a real fund could legitimately
have a zero balance and `fillna(0.0)` would make that indistinguishable
from "no data."

In [0]:
fund_financials_df = spark.table(silver_table("fund_financials"))
cash_current_df = spark.table(silver_table("cash_current"))

internal_funds_df = fund_financials_df.select(
    "fund_id", "available_cash_internal", "available_cash_external"
)

external_funds_df = (
    cash_current_df
    .select(F.col("internal_fund_id").alias("fund_id"))
    .distinct()
    .withColumn("_has_external_cash", F.lit(True))
)

cash_recon_df = (
    internal_funds_df
    .join(external_funds_df, on="fund_id", how="full_outer")
    .withColumn("_has_internal", F.col("available_cash_internal").isNotNull())
    .withColumn("_has_external_cash", F.coalesce(F.col("_has_external_cash"), F.lit(False)))
    .withColumn("available_cash_internal", F.coalesce(F.col("available_cash_internal"), F.lit(0.0)))
    .withColumn("available_cash_external", F.coalesce(F.col("available_cash_external"), F.lit(0.0)))
    .withColumn("difference", F.col("available_cash_internal") - F.col("available_cash_external"))
    .withColumn(
        "status",
        F.when(~F.col("_has_internal"), F.lit("MISSING_INTERNAL"))
         .when(~F.col("_has_external_cash"), F.lit("MISSING_EXTERNAL"))
         .when(F.abs(F.col("difference")) <= F.lit(CASH_TOLERANCE), F.lit("MATCH"))
         .otherwise(F.lit("BREAK"))
    )
    .withColumn(
        "break_reason",
        F.when(F.col("status") == "BREAK", F.lit("CASH_AMOUNT_MISMATCH"))
         .when(F.col("status") == "MISSING_EXTERNAL", F.lit("NO_EXTERNAL_CASH_FEED_FOR_FUND"))
         .when(F.col("status") == "MISSING_INTERNAL", F.lit("NO_INTERNAL_FINANCIALS_FOR_FUND"))
         .otherwise(F.lit(None).cast("string"))
    )
    .withColumn("recon_type", F.lit("CASH"))
    .withColumn("entity_id", F.lit(None).cast("string"))
    .withColumn("business_date", F.current_date())
    .withColumn(
        "recon_id",
        make_recon_id(F.lit("CASH"), F.col("fund_id"), F.lit(""), F.col("business_date").cast("string"))
    )
    .withColumn("source_a_value", F.col("available_cash_internal").cast("string"))
    .withColumn("source_b_value", F.col("available_cash_external").cast("string"))
    .withColumn("created_at", F.current_timestamp())
    .select(
        "recon_id", "recon_type", "business_date", "fund_id", "entity_id",
        "source_a_value", "source_b_value", "difference", "status",
        "break_reason", "created_at"
    )
)

cash_recon_df.show(truncate=False)

+----------------------------------------------------------------+----------+-------------+--------+---------+--------------------+--------------+--------------------+----------------+------------------------------+--------------------------+
|recon_id                                                        |recon_type|business_date|fund_id |entity_id|source_a_value      |source_b_value|difference          |status          |break_reason                  |created_at                |
+----------------------------------------------------------------+----------+-------------+--------+---------+--------------------+--------------+--------------------+----------------+------------------------------+--------------------------+
|3d59acacd4c5690aef8d4a59223a38febc6e30bb89cc1c2c4ea2f1af5ee12cbf|CASH      |2026-09-23   |FUND_004|NULL     |2.271448859E7       |0.0           |2.271448859E7       |MISSING_EXTERNAL|NO_EXTERNAL_CASH_FEED_FOR_FUND|2026-09-23 04:46:45.162559|
|8ae3485de931559d1210519c817

## B. POSITION reconciliation
Two independent checks

In [0]:
portfolio_valuation_df = spark.table(silver_table("portfolio_valuation"))

position_coverage_df = (
    portfolio_valuation_df
    .withColumn(
        "status",
        F.when(~F.col("has_position"), F.lit("MISSING_EXTERNAL")).otherwise(F.lit("MATCH"))
    )
    .withColumn(
        "break_reason",
        F.when(~F.col("has_position"), F.lit("NO_CUSTODIAN_POSITION_ON_FILE"))
         .otherwise(F.lit(None).cast("string"))
    )
    .withColumn("recon_type", F.lit("POSITION"))
    .withColumn("business_date", F.current_date())
    .withColumn(
        "recon_id",
        make_recon_id(F.lit("POSITION"), F.col("fund_id"), F.col("company_id"), F.col("business_date").cast("string"))
    )
    .withColumn("source_a_value", F.lit(None).cast("string"))  # no internal quantity source - see header note
    .withColumn("source_b_value", F.col("quantity").cast("string"))
    .withColumn("difference", F.lit(None).cast("double"))
    .withColumn("created_at", F.current_timestamp())
    .select(
        "recon_id", "recon_type", "business_date", "fund_id",
        F.col("company_id").alias("entity_id"),
        "source_a_value", "source_b_value", "difference", "status",
        "break_reason", "created_at"
    )
)

# --- POSITION_BREAK rows (external feed disagreeing with itself) ---
position_df = spark.table(silver_table("position"))
POSITION_KEY_COLS = ["internal_fund_id", "internal_asset_id", "business_date"]

pos_break_keys_df = (
    position_df
    .withColumn(
        "_pos_hash",
        F.sha2(F.concat_ws("||", F.col("quantity").cast("string"), F.col("price").cast("string")), 256)
    )
    .groupBy(*POSITION_KEY_COLS)
    .agg(F.countDistinct("_pos_hash").alias("_distinct_val_count"))
    .filter(F.col("_distinct_val_count") > 1)
)

position_break_df = (
    position_df
    .join(pos_break_keys_df.select(*POSITION_KEY_COLS), on=POSITION_KEY_COLS)
    .groupBy(*POSITION_KEY_COLS)
    .agg(F.min("quantity").alias("qty_a"), F.max("quantity").alias("qty_b"))
    .withColumn("status", F.lit("BREAK"))
    .withColumn("break_reason", F.lit("POSITION_BREAK"))
    .withColumn("recon_type", F.lit("POSITION"))
    .withColumn(
        "recon_id",
        make_recon_id(F.lit("POSITION"), F.col("internal_fund_id"), F.col("internal_asset_id"), F.col("business_date").cast("string"))
    )
    .withColumn("source_a_value", F.col("qty_a").cast("string"))
    .withColumn("source_b_value", F.col("qty_b").cast("string"))
    .withColumn("difference", F.col("qty_b") - F.col("qty_a"))
    .withColumn("created_at", F.current_timestamp())
    .select(
        "recon_id", "recon_type", "business_date",
        F.col("internal_fund_id").alias("fund_id"),
        F.col("internal_asset_id").alias("entity_id"),
        "source_a_value", "source_b_value", "difference", "status",
        "break_reason", "created_at"
    )
)

position_recon_df = position_coverage_df.unionByName(position_break_df)
position_recon_df.show(truncate=False)

+----------------------------------------------------------------+----------+-------------+--------+---------+--------------+--------------+----------+----------------+-----------------------------+--------------------------+
|recon_id                                                        |recon_type|business_date|fund_id |entity_id|source_a_value|source_b_value|difference|status          |break_reason                 |created_at                |
+----------------------------------------------------------------+----------+-------------+--------+---------+--------------+--------------+----------+----------------+-----------------------------+--------------------------+
|179f628888010d955ec643779e15e08a1a6403464f40dec6b357414c3327d2a8|POSITION  |2026-09-23   |FUND_005|PORT_0001|NULL          |NULL          |NULL      |MISSING_EXTERNAL|NO_CUSTODIAN_POSITION_ON_FILE|2026-09-23 04:46:52.760556|
|46dcd3adfbc7ad7e7b9f8da5a4e360a23b80eec09d576e5dad9c0f178e049550|POSITION  |2026-09-23   |FUND_

## C. REFERENCE reconciliation
Same two-check pattern as Position:

**C1 - Coverage**: portfolio companies with a `benchmark_ticker` but no
matching row in `reference_current` at all -> `MISSING_EXTERNAL`.

**C2 - Internal consistency**: `reference_current`'s own `changed` flag
(the AST005 currency-flip case, already computed day-over-day in `09`) ->
surfaced here as `BREAK`, same reasoning as Position's B2.

In [0]:
reference_current_df = spark.table(silver_table("reference_current"))
portfolio_company_df = spark.table(silver_table("portfolio_company"))

w_ref = Window.partitionBy("internal_asset_id").orderBy(F.col("business_date").desc())
latest_reference_df = (
    reference_current_df
    .withColumn("_rn", F.row_number().over(w_ref))
    .filter(F.col("_rn") == 1)
)

# --- C1: coverage - companies with a benchmark but no reference record at all
reference_coverage_df = (
    portfolio_company_df
    .filter(F.col("has_benchmark"))
    .join(
        latest_reference_df.select(F.col("internal_asset_id").alias("company_id")).withColumn("_has_reference", F.lit(True)),
        on="company_id", how="left"
    )
    .withColumn("_has_reference", F.coalesce(F.col("_has_reference"), F.lit(False)))
    .filter(~F.col("_has_reference"))
    .withColumn("status", F.lit("MISSING_EXTERNAL"))
    .withColumn("break_reason", F.lit("NO_REFERENCE_RECORD_ON_FILE"))
    .withColumn("recon_type", F.lit("REFERENCE"))
    .withColumn("business_date", F.current_date())
    .withColumn(
        "recon_id",
        make_recon_id(F.lit("REFERENCE"), F.col("fund_id"), F.col("company_id"), F.col("business_date").cast("string"))
    )
    .withColumn("source_a_value", F.lit(None).cast("string"))
    .withColumn("source_b_value", F.lit(None).cast("string"))
    .withColumn("difference", F.lit(None).cast("double"))
    .withColumn("created_at", F.current_timestamp())
    .select(
        "recon_id", "recon_type", "business_date", "fund_id",
        F.col("company_id").alias("entity_id"),
        "source_a_value", "source_b_value", "difference", "status",
        "break_reason", "created_at"
    )
)

# --- C2: internal consistency - the day-over-day `changed` flag ---
reference_break_df = (
    latest_reference_df
    .filter(F.col("changed") == True)
    .join(
        portfolio_company_df.select(F.col("company_id").alias("internal_asset_id"), "fund_id"),
        on="internal_asset_id", how="left"
    )
    .withColumn("status", F.lit("BREAK"))
    .withColumn("break_reason", F.lit("REFERENCE_ATTRIBUTE_CHANGED"))
    .withColumn("recon_type", F.lit("REFERENCE"))
    .withColumn(
        "recon_id",
        make_recon_id(F.lit("REFERENCE"), F.coalesce(F.col("fund_id"), F.lit("")), F.col("internal_asset_id"), F.col("business_date").cast("string"))
    )
    .withColumn("source_a_value", F.col("_prev_currency").cast("string"))
    .withColumn("source_b_value", F.col("currency").cast("string"))
    .withColumn("difference", F.lit(None).cast("double"))
    .withColumn("created_at", F.current_timestamp())
    .select(
        "recon_id", "recon_type", "business_date",
        F.coalesce(F.col("fund_id"), F.lit("UNKNOWN")).alias("fund_id"),
        F.col("internal_asset_id").alias("entity_id"),
        "source_a_value", "source_b_value", "difference", "status",
        "break_reason", "created_at"
    )
)

reference_recon_df = reference_coverage_df.unionByName(reference_break_df)
reference_recon_df.show(truncate=False)

+----------------------------------------------------------------+----------+-------------+--------+---------+--------------+--------------+----------+----------------+---------------------------+--------------------------+
|recon_id                                                        |recon_type|business_date|fund_id |entity_id|source_a_value|source_b_value|difference|status          |break_reason               |created_at                |
+----------------------------------------------------------------+----------+-------------+--------+---------+--------------+--------------+----------+----------------+---------------------------+--------------------------+
|37c8fd6c724c27e4fb7511ace09139729a167533776d4dc989f5d4bad9dc58b1|REFERENCE |2026-09-23   |FUND_005|PORT_0008|NULL          |NULL          |NULL      |MISSING_EXTERNAL|NO_REFERENCE_RECORD_ON_FILE|2026-09-23 04:46:58.083986|
|b6c9e3988497e93324c586e92f1afde23a3220518ea572e4c98334b67621d837|REFERENCE |2026-09-23   |FUND_005|PORT

## Assemble and write `silver.reconciliation`
One shared table across all three recon types, Uses a plain overwrite for now  - not yet a MERGE. 

In [0]:
reconciliation_df = (
    cash_recon_df
    .unionByName(position_recon_df)
    .unionByName(reference_recon_df)
)

write_silver(reconciliation_df, "reconciliation")
print(f"silver.reconciliation row count: {reconciliation_df.count()}")

print("Status breakdown by recon_type:")
reconciliation_df.groupBy("recon_type", "status").count().orderBy("recon_type", "status").show()

silver.reconciliation row count: 55
Status breakdown by recon_type:
+----------+----------------+-----+
|recon_type|          status|count|
+----------+----------------+-----+
|      CASH|           BREAK|    3|
|      CASH|MISSING_EXTERNAL|    2|
|  POSITION|           BREAK|    2|
|  POSITION|           MATCH|    1|
|  POSITION|MISSING_EXTERNAL|   29|
| REFERENCE|           BREAK|    1|
| REFERENCE|MISSING_EXTERNAL|   17|
+----------+----------------+-----+



### Summary
`silver.reconciliation` now holds CASH, POSITION and REFERENCE recon
results in one shared schema, ready for `Gold_Reconciliation`.


In [0]:
spark.table(silver_table("cash_current")).select("cash_id", "internal_fund_id", "amount", "business_date", "status").orderBy("internal_fund_id", "business_date").show(20, truncate=False)

+----------------+----------------+---------+-------------+-------+
|cash_id         |internal_fund_id|amount   |business_date|status |
+----------------+----------------+---------+-------------+-------+
|CASH20260915001 |FUND_001        |1.25E7   |2026-09-15   |SETTLED|
|CASH20260916001 |FUND_001        |1.2675E7 |2026-09-16   |SETTLED|
|CASH20260917001 |FUND_001        |1.285E7  |2026-09-17   |SETTLED|
|CASH20260917LATE|FUND_001        |250000.0 |2026-09-17   |SETTLED|
|CASH20260915002 |FUND_002        |8200000.0|2026-09-15   |SETTLED|
|CASH20260916002 |FUND_002        |8275000.0|2026-09-16   |SETTLED|
|CASH20260917002 |FUND_002        |8450000.0|2026-09-17   |SETTLED|
|CASH20260915003 |FUND_003        |5400000.0|2026-09-15   |SETTLED|
|CASH20260916003 |FUND_003        |5490000.0|2026-09-16   |SETTLED|
|CASH20260917003 |FUND_003        |5580000.0|2026-09-17   |SETTLED|
+----------------+----------------+---------+-------------+-------+



In [0]:
spark.table(silver_table("reconciliation")).filter(F.col("recon_type") == "CASH").select("fund_id", "source_a_value", "source_b_value", "difference", "status").show(truncate=False)

+--------+--------------------+--------------+--------------------+----------------+
|fund_id |source_a_value      |source_b_value|difference          |status          |
+--------+--------------------+--------------+--------------------+----------------+
|FUND_004|2.271448859E7       |0.0           |2.271448859E7       |MISSING_EXTERNAL|
|FUND_002|1.1459456980000004E7|8450000.0     |3009456.980000004   |BREAK           |
|FUND_003|2.0998249159999996E7|5580000.0     |1.5418249159999996E7|BREAK           |
|FUND_005|8944997.92          |0.0           |8944997.92          |MISSING_EXTERNAL|
|FUND_001|2.5361399970000003E7|1.31E7        |1.2261399970000003E7|BREAK           |
+--------+--------------------+--------------+--------------------+----------------+

